# Visualize PEFT LoRA Checkpoints

Use this notebook to inspect PEFT LoRA checkpoints produced by the SFT and GRPO/RFT notebooks in this repo. Set `checkpoint_dir` below, then run the notebook to load the model, render a `torchinfo` summary, and review the checkpoint metadata report.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from IPython.display import Markdown, display
from peft import PeftModel
from torchinfo import summary
from transformers import AutoModelForCausalLM, AutoTokenizer


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd()

checkpoint_dir = "sft-arithmetic-lora-demo/checkpoint-19"

device_map = "auto"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

summary_depth = 9
summary_verbose = 1
summary_col_names = ["input_size", "output_size", "num_params", "trainable"]
summary_row_settings = ["var_names", "depth"]

print(f"Repo root: {repo_root}")
print(f"Selected checkpoint_dir: {checkpoint_dir}")
print(f"torch_dtype: {torch_dtype}")
print(f"device_map: {device_map}")


Repo root: /Users/jim/Desktop/genai/rft-learning
Selected checkpoint_dir: sft-arithmetic-lora-demo/checkpoint-19
torch_dtype: torch.float32
device_map: auto


In [3]:
def discover_checkpoint_candidates(root: Path) -> list[Path]:
    return sorted(
        path for path in root.glob("*-demo/checkpoint-*")
        if (path / "adapter_config.json").exists()
    )


def load_json(path: Path) -> dict[str, Any]:
    with path.open() as fh:
        return json.load(fh)


def format_int(value: int | float | None) -> str:
    if value is None:
        return "-"
    return f"{int(value):,}"


def format_float(value: Any) -> str:
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def format_percent(part: int, whole: int) -> str:
    if whole == 0:
        return "0.00%"
    return f"{(part / whole) * 100:.2f}%"


def normalize_path(path_str: str, root: Path) -> Path:
    path = Path(path_str)
    return path if path.is_absolute() else (root / path)


def load_training_args_for_output_dir(root: Path, output_dir_name: str) -> tuple[Path | None, Any | None]:
    for candidate in sorted(root.glob("*-adapter/training_args.bin")):
        try:
            training_args = torch.load(candidate, map_location="cpu", weights_only=False)
        except Exception:
            continue

        if getattr(training_args, "output_dir", None) == output_dir_name:
            return candidate, training_args

    return None, None


def load_optimizer_state(path: Path | None) -> dict[str, Any] | None:
    if path is None or not path.exists():
        return None
    return torch.load(path, map_location="cpu")


def parameter_stats(model: torch.nn.Module) -> dict[str, int]:
    total = 0
    trainable = 0
    lora_only = 0

    for name, param in model.named_parameters():
        count = param.numel()
        total += count
        if param.requires_grad:
            trainable += count
        if "lora_" in name:
            lora_only += count

    return {
        "total": total,
        "trainable": trainable,
        "frozen": total - trainable,
        "lora_only": lora_only,
    }


def shared_parameter_count(model: torch.nn.Module) -> int:
    unique_total = sum(param.numel() for _, param in model.named_parameters())
    all_named_total = sum(param.numel() for _, param in model.named_parameters(remove_duplicate=False))
    return all_named_total - unique_total


def dtype_breakdown(model: torch.nn.Module) -> pd.DataFrame:
    counts: dict[str, int] = {}
    for _, param in model.named_parameters():
        key = str(param.dtype)
        counts[key] = counts.get(key, 0) + param.numel()

    rows = [
        {"dtype": dtype_name, "parameter_count": format_int(count)}
        for dtype_name, count in sorted(counts.items())
    ]
    return pd.DataFrame(rows)


def summarize_last_log_row(trainer_state: dict[str, Any] | None) -> str:
    if not trainer_state:
        return "-"

    log_history = trainer_state.get("log_history") or []
    if not log_history:
        return "-"

    last_row = log_history[-1]
    keys_to_skip = {"total_flos"}
    parts = []
    for key, value in last_row.items():
        if key in keys_to_skip:
            continue
        if isinstance(value, float):
            parts.append(f"{key}={value:.4f}")
        else:
            parts.append(f"{key}={value}")
    return ", ".join(parts)


checkpoint_candidates = discover_checkpoint_candidates(repo_root)

display(Markdown("## Discovered PEFT LoRA Checkpoint Candidates"))
display(pd.DataFrame({"checkpoint_dir": [str(path.relative_to(repo_root)) for path in checkpoint_candidates]}))


## Discovered PEFT LoRA Checkpoint Candidates

,checkpoint_dir
0,grpo-arithmetic-lora-demo/checkpoint-72
1,grpo-arithmetic-lora-early-stopping-demo/check...
2,grpo-arithmetic-lora-early-stopping-demo/check...
3,sft-arithmetic-lora-demo/checkpoint-19


In [4]:
checkpoint_path = normalize_path(checkpoint_dir, repo_root)

if not checkpoint_path.exists():
    candidate_list = "\n".join(f"- {path.relative_to(repo_root)}" for path in checkpoint_candidates)
    raise FileNotFoundError(
        "Checkpoint directory was not found. Set `checkpoint_dir` to one of:\n" + candidate_list
    )

adapter_config_path = checkpoint_path / "adapter_config.json"
if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"Expected a PEFT LoRA checkpoint with adapter_config.json at {adapter_config_path}"
    )

trainer_state_path = checkpoint_path / "trainer_state.json"
optimizer_path = checkpoint_path / "optimizer.pt"

adapter_config = load_json(adapter_config_path)
trainer_state = load_json(trainer_state_path) if trainer_state_path.exists() else None
optimizer_state = load_optimizer_state(optimizer_path if optimizer_path.exists() else None)

output_dir_name = checkpoint_path.parent.name
training_args_path, training_args = load_training_args_for_output_dir(repo_root, output_dir_name)

checkpoint_health = pd.DataFrame(
    [
        {"artifact": "adapter_config.json", "status": "found", "path": str(adapter_config_path.relative_to(repo_root))},
        {"artifact": "trainer_state.json", "status": "found" if trainer_state_path.exists() else "missing", "path": str(trainer_state_path.relative_to(repo_root))},
        {"artifact": "optimizer.pt", "status": "found" if optimizer_path.exists() else "missing", "path": str(optimizer_path.relative_to(repo_root))},
        {"artifact": "training_args.bin", "status": "found" if training_args_path else "missing", "path": str(training_args_path.relative_to(repo_root)) if training_args_path else "-"},
    ]
)

display(Markdown("## Checkpoint Health"))
display(checkpoint_health)

base_model_name = adapter_config.get("base_model_name_or_path")
if not base_model_name:
    raise ValueError("adapter_config.json does not contain base_model_name_or_path")

display(Markdown("## Resolved Base Model"))
display(pd.DataFrame([{"checkpoint_dir": str(checkpoint_path.relative_to(repo_root)), "base_model_name_or_path": base_model_name, "checkpoint_type": "PEFT LoRA"}]))


## Checkpoint Health

,artifact,status,path
0,adapter_config.json,found,sft-arithmetic-lora-demo/checkpoint-19/adapter...
1,trainer_state.json,found,sft-arithmetic-lora-demo/checkpoint-19/trainer...
2,optimizer.pt,found,sft-arithmetic-lora-demo/checkpoint-19/optimiz...
3,training_args.bin,found,sft-arithmetic-lora-adapter/training_args.bin


## Resolved Base Model

,checkpoint_dir,base_model_name_or_path,checkpoint_type
0,sft-arithmetic-lora-demo/checkpoint-19,Qwen/Qwen2.5-0.5B-Instruct,PEFT LoRA


In [5]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs: dict[str, Any] = {"dtype": torch_dtype}
if device_map is not None:
    load_kwargs["device_map"] = device_map

base_model = AutoModelForCausalLM.from_pretrained(base_model_name, **load_kwargs)
model = PeftModel.from_pretrained(base_model, str(checkpoint_path), is_trainable=True)
model.eval()

first_parameter = next(model.parameters())
model_device = first_parameter.device

example_batch = tokenizer(
    "What is 9 + 3?",
    return_tensors="pt",
    padding=False,
    truncation=True,
)
example_batch = {key: value.to(model_device) for key, value in example_batch.items()}

display(Markdown("## Loaded Model"))
display(pd.DataFrame([{
    "model_class": type(base_model).__name__,
    "peft_wrapper_class": type(model).__name__,
    "model_device": str(model_device),
}]))


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1828.60it/s]


## Loaded Model

,model_class,peft_wrapper_class,model_device
0,Qwen2ForCausalLM,PeftModelForCausalLM,mps:0


In [8]:
display(Markdown("## torchinfo Summary"))

display(Markdown("`torchinfo` may count shared or tied weights more than once. The report below uses unique parameter totals from `named_parameters()`."))

with torch.no_grad():
    model_summary = summary(
        model,
        input_data=example_batch,
        depth=summary_depth,
        col_names=summary_col_names,
        row_settings=summary_row_settings,
        verbose=0,
        device=str(model_device),
    )

model_summary


## torchinfo Summary

`torchinfo` may count shared or tied weights more than once. The report below uses unique parameter totals from `named_parameters()`.

Layer (type (var_name):depth-idx)                                                Input Shape               Output Shape              Param #                   Trainable
PeftModelForCausalLM (PeftModelForCausalLM)                                      --                        --                        --                        Partial
├─LoraModel (base_model): 1-1                                                    --                        --                        --                        Partial
│    └─Qwen2ForCausalLM (model): 2-1                                             --                        --                        --                        Partial
│    │    └─Qwen2Model (model): 3-1                                              --                        --                        --                        Partial
│    │    │    └─Embedding (embed_tokens): 4-1                                   [1, 8]                    [1, 8, 896]               (136,134,656)             Fals

In [7]:
stats = parameter_stats(model)
shared_params = shared_parameter_count(model)
active_adapters = getattr(model, "active_adapters", None)
if callable(active_adapters):
    active_adapters = active_adapters()

adapter_names = list(getattr(model, "peft_config", {}).keys()) if hasattr(model, "peft_config") else []

model_report = pd.DataFrame(
    [
        {"metric": "Selected checkpoint path", "value": str(checkpoint_path.relative_to(repo_root))},
        {"metric": "Checkpoint type", "value": "PEFT LoRA"},
        {"metric": "Base model", "value": base_model_name},
        {"metric": "Loaded model class", "value": type(base_model).__name__},
        {"metric": "PEFT wrapper class", "value": type(model).__name__},
        {"metric": "Total parameters", "value": format_int(stats['total'])},
        {"metric": "Trainable parameters", "value": f"{format_int(stats['trainable'])} ({format_percent(stats['trainable'], stats['total'])})"},
        {"metric": "Frozen parameters", "value": f"{format_int(stats['frozen'])} ({format_percent(stats['frozen'], stats['total'])})"},
        {"metric": "LoRA-only parameters", "value": format_int(stats['lora_only'])},
        {"metric": "Shared/tied parameters counted separately by torchinfo", "value": format_int(shared_params)},
        {"metric": "Adapter names", "value": ", ".join(adapter_names) if adapter_names else "-"},
        {"metric": "Active adapters", "value": ", ".join(active_adapters) if active_adapters else "-"},
    ]
)

peft_report = pd.DataFrame(
    [
        {"field": "peft_type", "value": adapter_config.get("peft_type", "-")},
        {"field": "task_type", "value": adapter_config.get("task_type", "-")},
        {"field": "r", "value": adapter_config.get("r", "-")},
        {"field": "lora_alpha", "value": adapter_config.get("lora_alpha", "-")},
        {"field": "lora_dropout", "value": adapter_config.get("lora_dropout", "-")},
        {"field": "target_modules", "value": ", ".join(adapter_config.get("target_modules", [])) or "-"},
        {"field": "inference_mode", "value": adapter_config.get("inference_mode", "-")},
    ]
)

training_report = pd.DataFrame(
    [
        {"field": "trainer_config_type", "value": type(training_args).__name__ if training_args is not None else "-"},
        {"field": "training_args_path", "value": str(training_args_path.relative_to(repo_root)) if training_args_path else "-"},
        {"field": "optimizer", "value": str(getattr(training_args, 'optim', '-')) if training_args is not None else "-"},
        {"field": "learning_rate", "value": format_float(getattr(training_args, 'learning_rate', None)) if training_args is not None else "-"},
        {"field": "scheduler", "value": str(getattr(training_args, 'lr_scheduler_type', '-')) if training_args is not None else "-"},
        {"field": "per_device_train_batch_size", "value": getattr(training_args, 'per_device_train_batch_size', '-') if training_args is not None else "-"},
        {"field": "gradient_accumulation_steps", "value": getattr(training_args, 'gradient_accumulation_steps', '-') if training_args is not None else "-"},
        {"field": "num_train_epochs", "value": getattr(training_args, 'num_train_epochs', '-') if training_args is not None else "-"},
        {"field": "save_steps", "value": getattr(training_args, 'save_steps', '-') if training_args is not None else "-"},
        {"field": "eval_strategy", "value": str(getattr(training_args, 'eval_strategy', '-')) if training_args is not None else "-"},
        {"field": "eval_steps", "value": getattr(training_args, 'eval_steps', '-') if training_args is not None else "-"},
        {"field": "load_best_model_at_end", "value": getattr(training_args, 'load_best_model_at_end', '-') if training_args is not None else "-"},
        {"field": "metric_for_best_model", "value": getattr(training_args, 'metric_for_best_model', '-') if training_args is not None else "-"},
        {"field": "greater_is_better", "value": getattr(training_args, 'greater_is_better', '-') if training_args is not None else "-"},
    ]
)

trainer_state_report = pd.DataFrame(
    [
        {"field": "global_step", "value": trainer_state.get('global_step', '-') if trainer_state else "-"},
        {"field": "epoch", "value": trainer_state.get('epoch', '-') if trainer_state else "-"},
        {"field": "max_steps", "value": trainer_state.get('max_steps', '-') if trainer_state else "-"},
        {"field": "best_model_checkpoint", "value": trainer_state.get('best_model_checkpoint', '-') if trainer_state else "-"},
        {"field": "best_metric", "value": format_float(trainer_state.get('best_metric')) if trainer_state else "-"},
        {"field": "log_history_rows", "value": len(trainer_state.get('log_history', [])) if trainer_state else "-"},
        {"field": "last_log_row", "value": summarize_last_log_row(trainer_state)},
    ]
)

optimizer_report = pd.DataFrame(
    [
        {"field": "optimizer_state_present", "value": optimizer_state is not None},
        {"field": "param_group_count", "value": len(optimizer_state.get('param_groups', [])) if optimizer_state else "-"},
        {"field": "first_group_hyperparameters", "value": json.dumps({k: v for k, v in optimizer_state.get('param_groups', [{}])[0].items() if k != 'params'}, default=str) if optimizer_state and optimizer_state.get('param_groups') else "-"},
    ]
)

display(Markdown("## Model Report"))
display(model_report)

display(Markdown("## PEFT LoRA Metadata"))
display(peft_report)

display(Markdown("## Training Metadata"))
display(training_report)

display(Markdown("## Trainer State Metadata"))
display(trainer_state_report)

display(Markdown("## Optimizer Metadata"))
display(optimizer_report)

display(Markdown("## Parameter Dtype Breakdown"))
display(dtype_breakdown(model))


## Model Report

,metric,value
0,Selected checkpoint path,sft-arithmetic-lora-demo/checkpoint-19
1,Checkpoint type,PEFT LoRA
2,Base model,Qwen/Qwen2.5-0.5B-Instruct
3,Loaded model class,Qwen2ForCausalLM
4,PEFT wrapper class,PeftModelForCausalLM
5,Total parameters,"502,437,760"
6,Trainable parameters,"8,404,992 (1.67%)"
7,Frozen parameters,"494,032,768 (98.33%)"
8,LoRA-only parameters,"8,404,992"
9,Shared/tied parameters counted separately by t...,"136,134,656"


## PEFT LoRA Metadata

,field,value
0,peft_type,LORA
1,task_type,CAUSAL_LM
2,r,16
3,lora_alpha,32
4,lora_dropout,0.05
5,target_modules,"q_proj, o_proj, gate_proj, down_proj, up_proj,..."
6,inference_mode,True


## Training Metadata

...

,field,value
0,global_step,19
1,epoch,1.0
2,max_steps,19
3,best_model_checkpoint,None
4,best_metric,-
5,log_history_rows,1
6,last_log_row,"entropy=0.0660, epoch=0.5333, grad_norm=0.0352..."


## Optimizer Metadata

,field,value
0,optimizer_state_present,True
1,param_group_count,2
2,first_group_hyperparameters,"{""weight_decay"": 0.0, ""lr"": 0.0, ""betas"": [0.9..."


## Parameter Dtype Breakdown

,dtype,parameter_count
0,torch.float32,"502,437,760"
